In [2]:
# One hot encoding

import numpy as np
def one_hot_encoding(sentence):
    words = sentence.lower().split()
    vocabulary = sorted(set(words))
    word_to_idx = {word: i for i, word in enumerate(vocabulary)}
    one_hot_matrix = np.zeros((len(words), len(vocabulary)))
    for i, word in enumerate(words):
        one_hot_matrix[i, word_to_idx[word]] = 1
    return one_hot_matrix, vocabulary

In [3]:
sentence = "Should we go to a pizzeria or do you prefer a restaurant?"
one_hot_matrix, vocabulary = one_hot_encoding(sentence)
print("Vocabulary:", vocabulary)
print("One-Hot Encoding Matrix:\n", one_hot_matrix)

Vocabulary: ['a', 'do', 'go', 'or', 'pizzeria', 'prefer', 'restaurant?', 'should', 'to', 'we', 'you']
One-Hot Encoding Matrix:
 [[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]]


In [4]:
# Bag of Words
# - Tokenize each document to get a list of words
# - Create a vocab of unique words and map each to the corresponding index of the vocab
# - Create a matrix where each row represents a document and each column, instead, a word in the vocabulary

In [33]:
def bag_of_words(sentences):
    tokenized_sentences = [sentence.lower().split() for sentence in sentences]
    flat_words = [word for sublist in tokenized_sentences for word in sublist]
    vocabulary = sorted(set(flat_words))
    word_to_idx = {word: i for i, word in enumerate(vocabulary)}
    bow_matrix = np.zeros((len(sentences), len(vocabulary)), dtype=int)
    for i, sentence in enumerate(tokenized_sentences):
        for word in sentence:
            if word in word_to_idx:
                bow_matrix[i, word_to_idx[word]] += 1
    return vocabulary, bow_matrix


In [34]:
corpus = ["This movie is awesome awesome",
          "I do not say is good, but neither awesome",
          "Awesome? Only a fool can say that"]
vocabulary, bow_matrix = bag_of_words(corpus)
print("Vocabulary:", vocabulary)
print("Bag of Words Matrix:\n", bow_matrix)

Vocabulary: ['a', 'awesome', 'awesome?', 'but', 'can', 'do', 'fool', 'good,', 'i', 'is', 'movie', 'neither', 'not', 'only', 'say', 'that', 'this']
Bag of Words Matrix:
 [[0 2 0 0 0 0 0 0 0 1 1 0 0 0 0 0 1]
 [0 1 0 1 0 1 0 1 1 1 0 1 1 0 1 0 0]
 [1 0 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0]]


In [35]:
len(vocabulary)

17

In [36]:
from sklearn.feature_extraction.text import CountVectorizer
def bag_of_words(sentences):
    vectorizer = CountVectorizer(tokenizer=lambda text: text.lower().split(), token_pattern=None)
    bow_matrix = vectorizer.fit_transform(sentences)
    return vectorizer.get_feature_names_out(), bow_matrix.toarray()

In [37]:
vocabulary, bow_matrix = bag_of_words(corpus)
print("Vocabulary:", vocabulary)
print("Bag of Words Matrix:\n", bow_matrix)

Vocabulary: ['a' 'awesome' 'awesome?' 'but' 'can' 'do' 'fool' 'good,' 'i' 'is' 'movie'
 'neither' 'not' 'only' 'say' 'that' 'this']
Bag of Words Matrix:
 [[0 2 0 0 0 0 0 0 0 1 1 0 0 0 0 0 1]
 [0 1 0 1 0 1 0 1 1 1 0 1 1 0 1 0 0]
 [1 0 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0]]


In [ ]:
len(vocabulary)

17

In [41]:
# TF-IDF

# tf = num repeated words in sentence / num words in sentence
from collections import Counter


def compute_tf(sentences):
    tokenized_sentences = [s.lower().split() for s in sentences]
    vocabulary = sorted(set(word for sentence in tokenized_sentences for word in sentence))
    word_idx = {word: i for i, word in enumerate(vocabulary)}
    
    tf = np.zeros((len(sentences), len(vocabulary)))
    for i, words in enumerate(tokenized_sentences):
        word_counts = Counter(words)
        total_words = len(words)
        if total_words > 0:
            # Extract indices and counts for only the unique words present in this sentence
            indices = [word_idx[word] for word in word_counts]
            tf[i, indices] = [count / total_words for count in word_counts.values()]
            
    return tf, vocabulary

In [ ]:
# IDF = ln(num_sentences / num_sentences containing words)
def compute_idf(sentences, vocabulary):
    num_documents = len(sentences)
    # Pre-tokenize sentences
    sentence_sets = [set(s.lower().split()) for s in sentences]
    # Allocated a 1D array for IDF (one value per word)
    idf = np.zeros(len(vocabulary))
    for idx, word in enumerate(vocabulary):
        df = sum(1 for s_set in sentence_sets if word in s_set)
        idf[idx] = np.log(num_documents / (1 + df))
    return idf

def tf_idf(sentences):
    tf, vocabulary = compute_tf(sentences)
    idf = compute_idf(sentences, vocabulary)
    tf_idf_matrix = tf * idf
    return vocabulary, tf_idf_matrix

In [45]:
vocabulary, tf_idf_matrix = tf_idf(corpus)
print("Vocabulary:", vocabulary)
print("TF-IDF Matrix:\n", tf_idf_matrix)

Vocabulary: ['a', 'awesome', 'awesome?', 'but', 'can', 'do', 'fool', 'good,', 'i', 'is', 'movie', 'neither', 'not', 'only', 'say', 'that', 'this']
TF-IDF Matrix:
 [[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.08109302 0.
  0.         0.         0.         0.         0.08109302]
 [0.         0.         0.         0.04505168 0.         0.04505168
  0.         0.04505168 0.04505168 0.         0.         0.04505168
  0.04505168 0.         0.         0.         0.        ]
 [0.05792359 0.         0.05792359 0.         0.05792359 0.
  0.05792359 0.         0.         0.         0.         0.
  0.         0.05792359 0.         0.05792359 0.        ]]


In [ ]:
# word2vec

